# 📈 Actividad 10: Reexploración Post-ETL
---
**Entrada:** `data/03_processed/master_dataset_fase1_v2.csv`  
**Salida:** 3 gráficos de validación en `data/04_reports/`

Gráficos:
1. **Serie temporal** — Producción vs Precio (doble eje)
2. **Heatmap de correlación** — Producción × Emergencias × Noticias
3. **Multivariable** — Producción vs Emergencias vs Noticias por mes


In [ ]:

import os, sys, json, warnings
import numpy as np, pandas as pd
import joblib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
with open('data/02_interim/pipeline_config.json','r',encoding='utf-8') as f:
    CFG = json.load(f)
DIRS=CFG['DIRS']; INTERIM=DIRS['interim']; PROCESSED=DIRS['processed']
REPORTS=DIRS['reports']; SCALERS=DIRS['scalers']
print(f"CWD: {os.getcwd()} | Config OK")


## 10.1 Cargar y Desnormalizar

In [ ]:

df = pd.read_csv(f"{PROCESSED}/master_dataset_fase1_v2.csv")
print(f"Dataset: {df.shape}")

scaler_path = f"{SCALERS}/scaler_fase1_v2.pkl"
cols_scaled = ['produccion_t','cosecha_ha','precio_chacra_kg',
               'num_emergencias','total_afectados','has_cultivo_perdidas','n_noticias']
cols_scaled = [c for c in cols_scaled if c in df.columns]

if os.path.exists(scaler_path):
    scaler = joblib.load(scaler_path)
    df_real = df.copy()
    df_real[cols_scaled] = scaler.inverse_transform(df[cols_scaled].fillna(0))
    print("Scaler cargado — valores desnormalizados")
else:
    df_real = df.copy()


## 10.2 Gráfico 1 — Producción vs Precio (Serie Temporal)

In [ ]:

trend = df_real.groupby('fecha_evento').agg(
    prod_total=('produccion_t','sum'), precio_prom=('precio_chacra_kg','mean')
).reset_index().sort_values('fecha_evento')

fig, ax1 = plt.subplots(figsize=(14,6))
ax1.plot(range(len(trend)), trend['prod_total'], 'g-o', markersize=3, linewidth=1.5, label='Producción (t)')
ax1.set_ylabel('Producción Total (t)', color='green', fontsize=11)
ax1.set_xlabel('Fecha', fontsize=11)
ax1.set_xticks(range(0,len(trend),6))
ax1.set_xticklabels(trend['fecha_evento'].iloc[::6], rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(range(len(trend)), trend['precio_prom'], 'r--s', markersize=3, linewidth=1.5, label='Precio (S/./kg)')
ax2.set_ylabel('Precio Chacra (S/./kg)', color='red', fontsize=11)

lines1,l1 = ax1.get_legend_handles_labels()
lines2,l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, l1+l2, loc='upper left')
fig.suptitle('Producción Total vs Precio Promedio del Limón (2021-2025)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{REPORTS}/g8_produccion_vs_precio.png", dpi=150, bbox_inches='tight')
plt.show()
print("[OK] g8_produccion_vs_precio.png")


## 10.3 Gráfico 2 — Heatmap de Correlación

In [ ]:

corr_cols = [c for c in ['produccion_t','precio_chacra_kg','num_emergencias',
             'total_afectados','n_noticias','month_sin','month_cos'] if c in df.columns]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10,8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(250,10,as_cmap=True),
            center=0, annot=True, fmt='.2f', square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlación: Producción × Emergencias × Noticias\n(Espacio reservado para variables NASA)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{REPORTS}/g9_correlacion_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("[OK] g9_correlacion_heatmap.png")


## 10.4 Gráfico 3 — Multivariable: Producción vs Emergencias vs Noticias

In [ ]:

multi = df_real.groupby('fecha_evento').agg(
    produccion=('produccion_t','sum'),
    emergencias=('num_emergencias','sum'),
    noticias=('n_noticias','first')
).reset_index().sort_values('fecha_evento')

fig, ax1 = plt.subplots(figsize=(14,6))
x = range(len(multi))

ax1.fill_between(x, multi['produccion'], alpha=0.3, color='green', label='Producción (t)')
ax1.plot(x, multi['produccion'], 'g-', linewidth=1)
ax1.set_ylabel('Producción (t)', fontsize=11, color='green')
ax1.set_xlabel('Fecha', fontsize=11)
ax1.set_xticks(range(0,len(multi),6))
ax1.set_xticklabels(multi['fecha_evento'].iloc[::6], rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.bar(x, multi['emergencias'], alpha=0.5, color='red', width=0.6, label='Emergencias')
ax2.set_ylabel('Nro. Emergencias', fontsize=11, color='red')

ax3 = ax1.twinx()
ax3.spines['right'].set_position(('axes', 1.12))
ax3.plot(x, multi['noticias'], 'b-^', markersize=4, linewidth=1.5, label='Noticias')
ax3.set_ylabel('Noticias/mes', fontsize=11, color='blue')

lines = []
for ax in [ax1, ax2, ax3]:
    l, lb = ax.get_legend_handles_labels()
    lines.extend(zip(l, lb))
ax1.legend(*zip(*lines), loc='upper left')
fig.suptitle('Análisis Multivariable: Producción × Emergencias × Noticias', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{REPORTS}/g10_multivariable.png", dpi=150, bbox_inches='tight')
plt.show()
print("[OK] g10_multivariable.png")


## 10.5 Resumen de Registros por Año

In [ ]:

print("Conteo de registros por año:")
for yr, cnt in df.groupby('anho').size().items():
    print(f"  {yr}: {cnt:,} filas")
print(f"\nTotal: {len(df):,} filas | {len(df.columns)} columnas")
print("\n[ACTIVIDAD 10] COMPLETADA.")
print("\n" + "="*60)
print("  PIPELINE FASE 1 — FINALIZADO EXITOSAMENTE")
print("="*60)


## TODO: INTEGRACIÓN DATA NASA (COMPAÑERO)
Añadir en el gráfico multivariable una línea de **precipitaciones** (eje derecho adicional):
```python
ax4 = ax1.twinx()
ax4.spines['right'].set_position(('axes', 1.24))
ax4.plot(x, multi['precipitacion_mm'], 'c-', linewidth=1.5, label='Precip. (mm)')
ax4.set_ylabel('Precipitación (mm)', color='cyan')
```
También agregar las variables climáticas al heatmap de correlación.
